# V8: continue P=10 from selected V7 checkpoints

This controlled run keeps the V7 architecture, training split, and loss fixed, and allows up to 100 more epochs from the selected V7 weights. V7 did not save optimizer state, so Adam restarts at 5e-5. V8 writes a resumable checkpoint to Drive after every epoch. Checkpoint selection uses validation only; test evaluation happens after selection.

Before running the training cell, choose **Runtime → Change runtime type → T4 GPU**.

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')
import os, torch
root = '/content/gdrive/MyDrive'
required = [
    f'{root}/DL_dataset_RSR25dB/{split}_P10.mat'
    for split in ('train', 'validation', 'test')
] + [
    f'{root}/unfolded_dl_v7_p10_extended_results/model_P10_seed{seed}_extended.pt'
    for seed in (20261026, 20261027, 20261028)
] + [
    f'{root}/DL_dataset_RSR25dB/v6_starting_models/gs_{split}_P10.json'
    for split in ('validation', 'test')
]
missing = [path for path in required if not os.path.isfile(path)]
assert not missing, f'Missing Drive files: {missing}'
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'not connected yet')
print('P=10 data, selected V7 weights, and GS curves found.')

In [ ]:
from google.colab import files
uploaded = files.upload()
required_scripts = {
    'colab_unfolded_dl_v3_highsnr.py',
    'colab_unfolded_dl_v5_snr_aware.py',
    'colab_unfolded_dl_v6_gs_gap.py',
    'colab_unfolded_dl_v7_continue_p10.py',
    'colab_unfolded_dl_v8_continue_p10.py',
}
missing = required_scripts.difference(uploaded)
assert not missing, f'Upload these scripts: {sorted(missing)}'
print('All five scripts uploaded.')

In [ ]:
assert torch.cuda.is_available(), 'Select and connect a T4 GPU runtime first.'
!python /content/colab_unfolded_dl_v8_continue_p10.py

In [ ]:
import csv, json, os
result_dir = '/content/gdrive/MyDrive/unfolded_dl_v8_continue_p10_results'
summary_path = f'{result_dir}/summary_unfolded_v8_p10_continuation.json'
with open(summary_path, encoding='utf-8') as f:
    results = json.load(f)
snrs = [-5, 0, 5, 10, 15, 20, 25, 30]
rows = []
for snr in snrs:
    key = str(snr)
    gaps = [r['test_dl_minus_gs_db_by_snr'][key] for r in results]
    dl = [r['test_metrics']['nmse_db_by_snr'][key] for r in results]
    rows.append({'SNR_dB': snr, 'V8_mean_DL_dB': sum(dl)/len(dl),
                 'V8_mean_DL_minus_GS_dB': sum(gaps)/len(gaps),
                 'Seeds_beating_GS': sum(g < 0 for g in gaps)})
for row in rows:
    print(f"{row['SNR_dB']:>3} dB | mean DL {row['V8_mean_DL_dB']:8.3f} dB | "
          f"DL-GS {row['V8_mean_DL_minus_GS_dB']:+.3f} dB | "
          f"beats GS {row['Seeds_beating_GS']}/3 seeds")
csv_path = f'{result_dir}/v8_test_mean_by_snr.csv'
with open(csv_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=rows[0].keys())
    writer.writeheader(); writer.writerows(rows)
print('Summary and checkpoints saved in:', result_dir)